# 05 - Colab / Kaggle: the same code at full scale

Runs the identical package on a GPU runtime at roughly 8x the shipped scale:
6000 training documents, 288 tokens per document, a wider encoder, and a longer
schedule. Nothing here is a different implementation -- only the config changes.

There is also an optional real-dataset cell. Read the caveat: FUNSD, CORD and
SROIE annotate *values*, not the token span each value was read from, so on real
data the grounding oracle can only be approximated by string matching. No number
in `docs/RESULTS.md` comes from that path.

## Install

In [ ]:
# In Colab or Kaggle:
# !git clone https://github.com/arslan-ahmad/grounded-document-extraction
# %cd grounded-document-extraction
# !pip install -q torch numpy scipy pandas pyyaml matplotlib
import sys, os
sys.path.insert(0, os.path.abspath("src") if os.path.isdir("src") else os.path.abspath("../src"))
import torch
print("torch", torch.__version__, "cuda", torch.cuda.is_available())

## Full-scale configuration

In [ ]:
from gdx.config import load_config
overrides = [
    "data.n_train=6000", "data.n_val=800", "data.n_test=1500",
    "data.max_tokens=288", "data.max_pages=3",
    "model.d_model=160", "model.n_layers=4", "model.n_heads=8", "model.d_ff=320",
    "model.dec_hidden=192",
    "optim.epochs=20", "optim.batch_size=32", "optim.lr=0.002",
    "run.n_threads=8",
]
cfg = load_config("configs/base.yaml", overrides)
print(cfg.to_dict()["model"])

## Train both heads and evaluate every arm

In [ ]:
from gdx.pipelines.experiments import run_seed
frame = run_seed(cfg, seed=0)
frame[["arm", "strict_accuracy", "canonical_accuracy", "coverage",
       "hallucination_rate", "grounding_exact"]]

## The invariant at scale

The guarantee is structural, so scale cannot break it. This asserts rather than
inspects.

In [ ]:
import pandas as pd
per_item = pd.read_csv("results/runs/seed0/per_item.csv")
span = per_item[per_item["arm"].str.startswith("span") & per_item["emitted"]]
print(f"{len(span)} emitted span values; ungrounded: {int((~span['grounded']).sum())}")
assert bool(span["grounded"].all())

## Optional: a real dataset, with the caveat

In [ ]:
# !python scripts/download_real.py --dataset cord   # prints the source and licence
# from gdx.data.real import load_real_dataset
# docs = load_real_dataset("cord", root="data/raw", split="test")
# print(len(docs), "documents; provenance recovered by string matching (approximate)")

## Ablations and analysis at scale

In [ ]:
from gdx.pipelines.ablations import run_model_ablations, run_verify_ablations, train_shared_span
shared = train_shared_span(cfg, 0)
run_verify_ablations(cfg, 0, shared=shared)
run_model_ablations(cfg, 0, shared=shared)
from gdx.pipelines.ablations import summarise_ablations
summarise_ablations()